In [3]:
import pandas as pd
import numpy as np

#  Charger le dataset
# On lit le fichier CSV contenant les données médicales
# Chaque ligne représente un patient
df = pd.read_csv("data.csv")

# Affichage des premières lignes pour vérifier
print("Aperçu du dataset :")
print(df.head())


# Nettoyage des données
# On supprime les colonnes inutiles pour l'apprentissage

# Supprimer la colonne vide si elle existe
if 'Unnamed: 32' in df.columns:
    df = df.drop(columns=['Unnamed: 32'])

# Supprimer la colonne id si elle existe
if 'id' in df.columns:
    df = df.drop(columns=['id'])

print("\nColonnes après nettoyage :")
print(df.columns)


# Transformer la variable cible
# La colonne diagnosis contient :
# M = Malignant
# B = Benign
#
# On transforme en valeurs numériques :
# M -> +1
# B -> -1
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': -1})


#  Séparer X et y
# X = variables explicatives
# y = variable cible

X = df.drop(columns=['diagnosis']).values.astype(float)
y = df['diagnosis'].values.astype(int)

print("\nDimensions de X :", X.shape)
print("Dimensions de y :", y.shape)

Aperçu du dataset :
         id diagnosis  radius_mean  texture_mean  perimeter_mean  area_mean  \
0    842302         M        17.99         10.38          122.80     1001.0   
1    842517         M        20.57         17.77          132.90     1326.0   
2  84300903         M        19.69         21.25          130.00     1203.0   
3  84348301         M        11.42         20.38           77.58      386.1   
4  84358402         M        20.29         14.34          135.10     1297.0   

   smoothness_mean  compactness_mean  concavity_mean  concave points_mean  \
0          0.11840           0.27760          0.3001              0.14710   
1          0.08474           0.07864          0.0869              0.07017   
2          0.10960           0.15990          0.1974              0.12790   
3          0.14250           0.28390          0.2414              0.10520   
4          0.10030           0.13280          0.1980              0.10430   

   ...  texture_worst  perimeter_worst  ar

In [6]:

#  Normalisation des données
# La normalisation est très importante pour SVM
# car les variables n'ont pas la même échelle.
#
# Exemple :
# area_mean peut être très grand
# smoothness_mean est très petit
#
# Sans normalisation, les grandes valeurs dominent le modèle.

X_mean = np.mean(X, axis=0)
X_std = np.std(X, axis=0)

# Formule :
# X_normalisé = (X - moyenne) / écart-type

X = (X - X_mean) / X_std

# Séparer Train et Test 

# Fixer seed pour avoir toujours le même mélange
np.random.seed(42)

# Créer des indices
indices = np.arange(X.shape[0])

# Mélanger les indices
np.random.shuffle(indices)

# Réordonner X et y avec les indices mélangés
X = X[indices]
y = y[indices]

# 80% pour entraînement
# 20% pour test
train_size = int(0.8 * len(X))

X_train = X[:train_size]
y_train = y[:train_size]

X_test = X[train_size:]
y_test = y[train_size:]

# Initialisation des paramètres SVM
# Nombre d'exemples et nombre de variables
n_samples, n_features = X_train.shape

# w représente les poids du modèle
# Il y a un poids pour chaque variable
w = np.zeros(n_features)

# b représente le biais
b = 0

#  Hyperparamètres
# learning_rate contrôle la vitesse d'apprentissage
learning_rate = 0.001

# lambda_param contrôle la régularisation
# Il empêche les poids w de devenir trop grands
lambda_param = 0.01

# Nombre de passages sur les données
epochs = 1000

#  Entraînement du SVM

for epoch in range(epochs):

    # Parcourir chaque patient du dataset d'entraînement
    for i in range(n_samples):

        # Exemple courant
        x_i = X_train[i]

        # Classe réelle de cet exemple
        y_i = y_train[i]

        # Calcul du score linéaire :
        # score = w.X + b
        score = np.dot(x_i, w) + b

        # Condition de marge SVM :
        #
        # y_i * (w.X + b) >= 1
        #
        # Si cette condition est vraie :
        # le point est bien classé et en dehors de la marge.
        #
        # Si elle est fausse :
        # le point est mal classé ou trop proche de la frontière.

        condition = y_i * score

        if condition >= 1:

            # Cas 1 :
            # Le point est bien classé.
            #
            # On met à jour seulement avec la régularisation.
            #
            # Objectif :
            # réduire la taille de w pour garder une bonne marge.

            dw = 2 * lambda_param * w
            db = 0

        else:

            # Cas 2 :
            # Le point est mal classé ou dans la marge.
            #
            # On corrige w et b pour mieux séparer les classes.

            dw = 2 * lambda_param * w - y_i * x_i
            db = -y_i

        # Mise à jour des poids
        #
        # Formule générale :
        # paramètre = paramètre - learning_rate * gradient

        w = w - learning_rate * dw

        # Mise à jour du biais
        b = b - learning_rate * db

# . Fonction de prédiction

def predict(X):
    # Calcul du score :
    # score = w.X + b
    linear_output = np.dot(X, w) + b

    # Si score >= 0 :
    # classe prédite = +1
    #
    # Sinon :
    # classe prédite = -1
    return np.where(linear_output >= 0, 1, -1)

#Prédictions sur les données de test

y_pred = predict(X_test)

# Affichage des résultats

print("Poids w :")
print(w)

print("\nBiais b :")
print(b)

print("\n20 vraies classes :")
print(y_test[:20])

print("\n20 classes prédites :")
print(y_pred[:20])



Poids w :
[ 0.19789681  0.25973968  0.18414648  0.20423804  0.09655069 -0.11030273
  0.32576037  0.32239263 -0.04430586 -0.15743601  0.3795639  -0.15428411
  0.36231742  0.29515756  0.02167317 -0.28630322 -0.04937175  0.01131774
 -0.00622885 -0.20899424  0.3179576   0.46395748  0.3111795   0.31701913
  0.3435502  -0.06657601  0.33844616  0.28795051  0.43143303  0.16073432]

Biais b :
-0.2600000000000002

20 vraies classes :
[ 1  1 -1 -1  1 -1 -1 -1  1  1  1 -1 -1  1 -1 -1 -1  1 -1  1]

20 classes prédites :
[ 1 -1 -1 -1  1 -1 -1 -1  1  1  1 -1 -1  1 -1 -1 -1  1 -1  1]
